# Etapa 3 — Construção de Modelo (K-Means Clustering)

## Projeto: Análise da Variabilidade Regional do Mercado Automotivo Brasileiro via RENAVAM

**Objetivo:** Aplicar o algoritmo K-Means para descobrir agrupamentos naturais ("Brasis Automotivos") baseados na distribuição percentual da frota e indicadores socioeconômicos dos municípios.

**Justificativa da escolha:** O K-Means foi selecionado como primeiro modelo por ser uma técnica de aprendizado não supervisionado que permite descobrir padrões ocultos na estrutura dos dados, sem necessidade de variável-alvo pré-definida. Essa abordagem exploratória é ideal como primeiro passo antes de modelos supervisionados (Etapa 4), pois revela agrupamentos naturais que podem servir como features adicionais.

## 1. Pré-processamento dos Dados

### 1.1. Estratégia de Limpeza
- Exclusão de municípios com `populacao=0` (dados ausentes mascarados pelo ETL)
- Exclusão de municípios com `TOTAL=0` (sem frota registrada)

### 1.2. Seleção de Features para Clusterização
Utilizamos variáveis que representam o **perfil socioeconômico** e a **composição da frota**:
- `target_perc_diesel` — % da frota que é diesel
- `target_perc_utilitarios` — % da frota que é utilitários
- `pib_agro_por_habitante` — vocação agropecuária
- `pib_per_capita` — riqueza geral
- `densidade_demografica` — urbanização
- `presenca_rodovia_federal` — polo logístico

### 1.3. Normalização
O K-Means é sensível à escala das variáveis (usa distância euclidiana). Aplicamos `StandardScaler` para que todas as features tenham média=0 e desvio=1.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style="whitegrid")

# Carregar dataset
df = pd.read_csv('ETL/Dados Tratados/Dados Unificados/dataset_final_modelagem.csv')
print(f"Dataset original: {df.shape[0]} munic\u00edpios")

In [ ]:
# 1. Limpeza: remover munic\u00edpios com dados ausentes mascarados
df_clean = df[(df['populacao'] > 0) & (df['TOTAL'] > 0)].copy()
print(f"Ap\u00f3s limpeza: {df_clean.shape[0]} munic\u00edpios (removidos: {df.shape[0] - df_clean.shape[0]})")

# 2. Sele\u00e7\u00e3o de features para clusteriza\u00e7\u00e3o
features_cluster = [
    'target_perc_diesel',
    'target_perc_utilitarios',
    'pib_agro_por_habitante',
    'pib_per_capita',
    'densidade_demografica',
    'presenca_rodovia_federal'
]

X = df_clean[features_cluster].copy()

# 3. Log transform em vari\u00e1veis com assimetria extrema (identificada na Etapa 2)
for col in ['pib_agro_por_habitante', 'pib_per_capita', 'densidade_demografica']:
    X[col] = np.log1p(X[col])  # log(1+x) para evitar log(0)

print(f"\nFeatures selecionadas: {features_cluster}")
print(f"Shape final: {X.shape}")

# 4. Normaliza\u00e7\u00e3o com StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nAp\u00f3s normaliza\u00e7\u00e3o:")
print(f"  M\u00e9dia das features: {X_scaled.mean(axis=0).round(4)}")
print(f"  Desvio das features: {X_scaled.std(axis=0).round(4)}")

## 2. Determinação do Número Ótimo de Clusters

Antes de rodar o K-Means, precisamos definir o número de clusters (k). Utilizamos dois métodos complementares:

1. **Método do Cotovelo (Elbow Method):** Plota a inércia (soma das distâncias ao centróide) para diferentes valores de k. O "cotovelo" indica o ponto onde adicionar mais clusters traz retornos decrescentes.
2. **Silhouette Score:** Mede quão bem cada ponto se encaixa no seu cluster vs. o cluster vizinho. Varia de -1 (mal classificado) a +1 (bem classificado). Quanto maior, melhor.

In [ ]:
# Teste de k de 2 a 10
k_range = range(2, 11)
inertias = []
silhouettes = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

# Gr\u00e1ficos lado a lado
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Elbow
axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('N\u00famero de Clusters (k)')
axes[0].set_ylabel('In\u00e9rcia')
axes[0].set_title('M\u00e9todo do Cotovelo (Elbow Method)')
axes[0].set_xticks(list(k_range))

# Silhouette
axes[1].plot(k_range, silhouettes, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('N\u00famero de Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score por k')
axes[1].set_xticks(list(k_range))

plt.tight_layout()
plt.show()

# Melhor k pelo Silhouette
best_k = list(k_range)[np.argmax(silhouettes)]
print(f"\nMelhor k pelo Silhouette Score: {best_k} (score={max(silhouettes):.4f})")
print(f"\nTodos os scores:")
for k, s in zip(k_range, silhouettes):
    marker = " <-- melhor" if k == best_k else ""
    print(f"  k={k}: Silhouette={s:.4f} | In\u00e9rcia={inertias[k-2]:.0f}{marker}")

## 3. Treinamento do Modelo K-Means

### Sobre o Algoritmo
O K-Means é um algoritmo de aprendizado não supervisionado que particiona os dados em k grupos, minimizando a variância intra-cluster (inércia). Funciona iterativamente:
1. Inicializa k centróides aleatoriamente
2. Atribui cada ponto ao centróide mais próximo
3. Recalcula os centróides como a média dos pontos do cluster
4. Repete até convergência

**Vantagens:** Eficiente em grandes datasets, interpretável, escalável.
**Limitações:** Sensível à inicialização (mitigado com `n_init=10`), assume clusters esféricos, requer definição prévia de k.

### Parâmetros Escolhidos
- `n_clusters`: definido pelo Silhouette Score
- `random_state=42`: reprodutibilidade
- `n_init=10`: executa 10 inicializações e escolhe a melhor
- `max_iter=300`: limite de iterações para convergência

In [ ]:
# Treinar modelo final com o melhor k
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10, max_iter=300)
df_clean['cluster'] = kmeans_final.fit_predict(X_scaled)

print(f"Modelo K-Means treinado com k={best_k}")
print(f"In\u00e9rcia final: {kmeans_final.inertia_:.2f}")
print(f"Itera\u00e7\u00f5es at\u00e9 converg\u00eancia: {kmeans_final.n_iter_}")
print(f"\nDistribui\u00e7\u00e3o dos clusters:")
print(df_clean['cluster'].value_counts().sort_index())
print(f"\nSilhouette Score final: {silhouette_score(X_scaled, df_clean['cluster']):.4f}")

## 4. Análise e Interpretação dos Clusters

O objetivo principal é entender **o que define cada cluster** — quais características socioeconômicas e de frota distinguem os grupos de municípios.

In [ ]:
# Perfil m\u00e9dio de cada cluster
perfil_medio = df_clean.groupby('cluster')[features_cluster].mean().round(4)
print("PERFIL M\u00c9DIO DE CADA CLUSTER")
print("=" * 80)
print(perfil_medio.T)

# Contar munic\u00edpios e UFs por cluster
print("\n\nDISTRIBUI\u00c7\u00c3O GEOGR\u00c1FICA POR CLUSTER")
print("=" * 80)
for c in sorted(df_clean['cluster'].unique()):
    subset = df_clean[df_clean['cluster'] == c]
    top_ufs = subset['uf'].value_counts().head(5)
    print(f"\nCluster {c} ({len(subset)} munic\u00edpios):")
    print(f"  Popula\u00e7\u00e3o m\u00e9dia: {subset['populacao'].mean():,.0f}")
    print(f"  % Diesel m\u00e9dio: {subset['target_perc_diesel'].mean():.2f}%")
    print(f"  PIB Agro/hab m\u00e9dio: R$ {subset['pib_agro_por_habitante'].mean():,.0f}")
    print(f"  Top 5 UFs: {dict(top_ufs)}")

In [ ]:
# Visualiza\u00e7\u00e3o dos clusters em 2D
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors = sns.color_palette('Set2', n_colors=best_k)

# 1. PIB Agro vs % Diesel
for c in sorted(df_clean['cluster'].unique()):
    mask = df_clean['cluster'] == c
    axes[0].scatter(df_clean.loc[mask, 'pib_agro_por_habitante'],
                    df_clean.loc[mask, 'target_perc_diesel'],
                    c=[colors[c]], label=f'Cluster {c}', alpha=0.4, s=15)
axes[0].set_xlabel('PIB Agro por Habitante (R$)')
axes[0].set_ylabel('% Diesel')
axes[0].set_title('PIB Agropecu\u00e1rio vs % Diesel')
axes[0].legend()

# 2. Densidade vs % Utilit\u00e1rios
for c in sorted(df_clean['cluster'].unique()):
    mask = df_clean['cluster'] == c
    axes[1].scatter(df_clean.loc[mask, 'densidade_demografica'],
                    df_clean.loc[mask, 'target_perc_utilitarios'],
                    c=[colors[c]], label=f'Cluster {c}', alpha=0.4, s=15)
axes[1].set_xlabel('Densidade Demogr\u00e1fica')
axes[1].set_ylabel('% Utilit\u00e1rios')
axes[1].set_title('Densidade vs % Utilit\u00e1rios')
axes[1].set_xlim(0, 1000)
axes[1].legend()

# 3. Boxplot de % Diesel por cluster
sns.boxplot(data=df_clean, x='cluster', y='target_perc_diesel',
            palette='Set2', ax=axes[2])
axes[2].set_title('Distribui\u00e7\u00e3o de % Diesel por Cluster')
axes[2].set_xlabel('Cluster')
axes[2].set_ylabel('% Diesel')

plt.suptitle('Visualiza\u00e7\u00e3o dos Clusters Encontrados', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Gr\u00e1fico de Silhouette por cluster
fig, ax = plt.subplots(figsize=(10, 8))

sample_silhouette = silhouette_samples(X_scaled, df_clean['cluster'])
y_lower = 10

for c in sorted(df_clean['cluster'].unique()):
    cluster_values = sample_silhouette[df_clean['cluster'] == c]
    cluster_values.sort()
    
    size = len(cluster_values)
    y_upper = y_lower + size
    
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_values,
                     alpha=0.7, color=colors[c], label=f'Cluster {c}')
    ax.text(-0.05, y_lower + 0.5 * size, str(c))
    y_lower = y_upper + 10

avg_score = silhouette_score(X_scaled, df_clean['cluster'])
ax.axvline(x=avg_score, color='red', linestyle='--', label=f'M\u00e9dia={avg_score:.3f}')
ax.set_xlabel('Silhouette Score')
ax.set_ylabel('Cluster')
ax.set_title('Silhouette Plot \u2014 Qualidade Individual por Cluster')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Silhouette Score m\u00e9dio por cluster:")
for c in sorted(df_clean['cluster'].unique()):
    score = sample_silhouette[df_clean['cluster'] == c].mean()
    print(f"  Cluster {c}: {score:.4f}")

## 5. Nomeação e Interpretação dos Clusters

Com base no perfil médio de cada cluster, atribuímos nomes descritivos que representam os "Brasis Automotivos" descobertos pelo modelo.

In [ ]:
# Interpreta\u00e7\u00e3o autom\u00e1tica baseada nas m\u00e9dias dos clusters
print("INTERPRETA\u00c7\u00c3O DOS CLUSTERS \u2014 Brasis Automotivos")
print("=" * 80)

for c in sorted(df_clean['cluster'].unique()):
    subset = df_clean[df_clean['cluster'] == c]
    diesel_medio = subset['target_perc_diesel'].mean()
    agro_medio = subset['pib_agro_por_habitante'].mean()
    pop_media = subset['populacao'].mean()
    dens_media = subset['densidade_demografica'].mean()
    n = len(subset)
    
    if diesel_medio > 12 and agro_medio > 10000:
        nome = "Brasil Agroindustrial (Alto Diesel + Alto PIB Agro)"
    elif diesel_medio > 10:
        nome = "Brasil Rural/Pecu\u00e1rio (Diesel acima da m\u00e9dia)"
    elif pop_media > 100000:
        nome = "Brasil Metropolitano (Grandes centros urbanos)"
    elif dens_media > 200:
        nome = "Brasil Urbano M\u00e9dio (Cidades m\u00e9dias, frota diversificada)"
    else:
        nome = "Brasil Municipal T\u00edpico (Pequenos munic\u00edpios)"
    
    print(f"\nCluster {c}: {nome}")
    print(f"  Munic\u00edpios: {n} ({n/len(df_clean)*100:.1f}% do total)")
    print(f"  % Diesel m\u00e9dio: {diesel_medio:.2f}%")
    print(f"  PIB Agro/hab: R$ {agro_medio:,.0f}")
    print(f"  Popula\u00e7\u00e3o m\u00e9dia: {pop_media:,.0f}")
    print(f"  Densidade m\u00e9dia: {dens_media:,.0f} hab/km\u00b2")

## 6. Avaliação do Modelo — Métricas

### Métrica Principal: Silhouette Score
O **Silhouette Score** foi escolhido como métrica principal porque:
- Mede a qualidade da separação entre clusters (coesão interna vs. separação externa)
- Não requer variável-alvo (adequado para aprendizado não supervisionado)
- Varia de -1 a +1, com interpretação direta:
  - **> 0.5:** Estrutura forte de clusters
  - **0.25 a 0.5:** Estrutura razoável
  - **< 0.25:** Estrutura fraca ou sobreposta

### Métrica Complementar: Inércia (Elbow Method)
A inércia mede a soma das distâncias quadradas de cada ponto ao centróide do seu cluster. Menor inércia = clusters mais compactos.

In [ ]:
# Resumo final das m\u00e9tricas
print("RESUMO DE AVALIA\u00c7\u00c3O DO MODELO K-Means")
print("=" * 60)
print(f"N\u00famero de clusters (k): {best_k}")
print(f"Silhouette Score: {silhouette_score(X_scaled, df_clean['cluster']):.4f}")
print(f"In\u00e9rcia: {kmeans_final.inertia_:.2f}")
print(f"Itera\u00e7\u00f5es: {kmeans_final.n_iter_}")
print(f"Munic\u00edpios analisados: {len(df_clean)}")
print(f"Features utilizadas: {len(features_cluster)}")

score = silhouette_score(X_scaled, df_clean['cluster'])
if score > 0.5:
    qualidade = "FORTE \u2014 clusters bem separados"
elif score > 0.25:
    qualidade = "RAZO\u00c1VEL \u2014 clusters com alguma sobreposi\u00e7\u00e3o"
else:
    qualidade = "FRACA \u2014 clusters muito sobrepostos"
print(f"\nInterpreta\u00e7\u00e3o: Estrutura de clusters {qualidade}")

## 7. Pipeline de Pesquisa e Análise de Dados

O pipeline abaixo documenta todas as etapas realizadas, desde a especificação do problema até a avaliação do modelo.

```
1. ESPECIFICAÇÃO DO PROBLEMA
   Questão: Como segmentar municípios por perfil de frota?
   Tipo: Aprendizado Não Supervisionado (Clusterização)

2. COLETA DE DADOS
   Fontes: SENATRAN (frota), IBGE (PIB/Censo), DNIT (rodovias)
   Granularidade: Municipal (5.571 municípios)

3. PRÉ-PROCESSAMENTO
   - Limpeza: remoção de pop=0 e TOTAL=0
   - Feature Engineering: proporções (%), binarização DNIT
   - Transformação: log1p em variáveis assimétricas
   - Normalização: StandardScaler (média=0, desvio=1)

4. SELEÇÃO DO MODELO
   Algoritmo: K-Means
   Justificativa: exploratório, escalável, interpretável

5. OTIMIZAÇÃO DE HIPERPARÂMETROS
   - Método do Cotovelo (Inércia vs k)
   - Silhouette Score para cada k (2 a 10)
   - k ótimo selecionado pelo maior Silhouette

6. TREINAMENTO E AVALIAÇÃO
   - Métrica principal: Silhouette Score
   - Métrica complementar: Inércia
   - Análise de perfil por cluster

7. INTERPRETAÇÃO E DOCUMENTAÇÃO
   - Nomeação dos clusters ("Brasis Automotivos")
   - Visualizações: scatter, boxplot, silhouette plot
   - Conexão com objetivos do projeto
```

## 8. Conclusão da Etapa 3

O modelo K-Means permitiu descobrir agrupamentos naturais na frota automotiva brasileira, segmentando os municípios em perfis distintos que refletem diferenças socioeconômicas reais. As principais contribuições desta etapa:

1. **Validação empírica** de que o Brasil possui "perfis automotivos" distintos, conforme hipotetizado na Etapa 1
2. **Identificação de variáveis-chave** que discriminam os grupos (PIB agro, % diesel)
3. **Base para a Etapa 4**, onde os clusters poderão ser usados como feature adicional nos modelos supervisionados

### Próximos Passos (Etapa 4)
- Implementar **Regressão Linear Múltipla** para prever `target_perc_diesel`
- Implementar **Random Forest Regressor** para a mesma tarefa
- Comparar os 3 modelos com métricas de regressão (MAE, RMSE, R²)
- Avaliar se a adição do cluster como feature melhora os modelos supervisionados